In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import random
from time import sleep
import pandas as pd
from dotenv import load_dotenv
import os
import re

In [ ]:
# --- 1. CONFIGURACIÓN Y CARGA DE DATOS (TUS CSV) ---
def cargar_catalogos():
    try:
        df_marcas = pd.read_csv('../datasets/seed_marcas.csv', sep=';', encoding='utf-8-sig')
        df_versiones = pd.read_csv('../datasets/seed_versiones.csv', sep=';', encoding='utf-8-sig')
        
        # Diccionarios para búsqueda rápida
        mapa_marcas = dict(zip(df_marcas['nombre_marca'].str.lower(), df_marcas['marca_id']))
        
        # Mapeo de versiones: El nombre completo del catálogo es la clave
        mapa_versiones = dict(zip(df_versiones['nombre_completo'].str.lower(), df_versiones['version_id']))
        
        return mapa_marcas, mapa_versiones, df_marcas['nombre_marca'].tolist()
    except FileNotFoundError:
        print("Error: No se encontraron los archivos CSV del catálogo.")
        return {}, {}, []

mapa_marcas, mapa_versiones, lista_nombres_marcas = cargar_catalogos()

In [3]:
load_dotenv()
WEB_BASE = os.getenv("WEB_BASE")
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json"
}

In [ ]:
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
lista_coches = []

def normalizar(texto):
    return texto.lower().strip()

In [ ]:
# --- 3. BUCLE PRINCIPAL DE SCRAPING ---
try:
    for page in range(1, 313):
        url = f'{WEB_BASE}comprar-coche/?page={page}'
        driver.get(url)
        sleep(random.uniform(2, 4)) 

        cars = driver.find_elements(By.CLASS_NAME, 'root___Dz4kU')
        print(f"Página {page}: Procesando {len(cars)} anuncios...")

        for car in cars:
            try:
                # Recuperar Título
                nombre_completo = car.find_element(By.CLASS_NAME, 'title___uRijL').text
                nombre_norm = normalizar(nombre_completo)

                # A. Identificar Marca y su ID
                marca_encontrada = "Desconocida"
                marca_id = None
                for m_nombre in lista_nombres_marcas:
                    if m_nombre.lower() in nombre_norm:
                        marca_encontrada = m_nombre
                        marca_id = mapa_marcas.get(m_nombre.lower())
                        break

                # B. Identificar Versión y su ID (Búsqueda por coincidencia)
                version_id = None
                # Buscamos si algún nombre_completo de nuestro catálogo está en el título del anuncio
                for v_nombre_cat, v_id in mapa_versiones.items():
                    if v_nombre_cat in nombre_norm:
                        version_id = v_id
                        break

                # C. Extraer Datos Técnicos con limpieza Regex
                km_text = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="mileage"]').text
                km_final = int(re.sub(r'\D', '', km_text))

                price_text = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="price"]').text
                price_final = int(re.sub(r'\D', '', price_text))

                power_text = car.find_element(By.CSS_SELECTOR, '[data-qa-selector="horsePower"]').text
                try:
                    power_cv = int(re.search(r'\((\d+)\sCV\)', power_text).group(1))
                except:
                    power_cv = 0

                # D. Construcción del Objeto
                datos_coche = {
                    "marca_nombre": marca_encontrada,
                    "marca_id": marca_id,
                    "version_id_catalogo": version_id,
                    "titulo_anuncio": nombre_completo,
                    "registro": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="registration"]').text,
                    "km": km_final,
                    "cambio": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="transmission"]').text,
                    "combustible": car.find_element(By.CSS_SELECTOR, '[data-qa-selector="fuelType"]').text,
                    "cv": power_cv,
                    "precio": price_final
                }
                
                lista_coches.append(datos_coche)

            except Exception as e:
                continue

except Exception as e:
    print(f"Fallo crítico: {e}")

finally:
    # --- 4. EXPORTACIÓN FINAL ---
    driver.quit()
    df_final = pd.DataFrame(lista_coches)
    df_final.to_csv('../datasets/data_compramostucoche.csv', index=False, sep=';', encoding='utf-8-sig')
    print(f"{len(lista_coches)} coches guardados.")

Página 1: Procesando 10 anuncios...
Página 2: Procesando 10 anuncios...
Página 3: Procesando 10 anuncios...
Página 4: Procesando 10 anuncios...
Página 5: Procesando 10 anuncios...
Página 6: Procesando 10 anuncios...
Página 7: Procesando 10 anuncios...
Página 8: Procesando 10 anuncios...
Página 9: Procesando 10 anuncios...
Página 10: Procesando 10 anuncios...
Página 11: Procesando 10 anuncios...
Página 12: Procesando 10 anuncios...
Página 13: Procesando 10 anuncios...
Página 14: Procesando 10 anuncios...
Página 15: Procesando 10 anuncios...
Página 16: Procesando 10 anuncios...
Página 17: Procesando 10 anuncios...
Página 18: Procesando 10 anuncios...
Página 19: Procesando 10 anuncios...
Página 20: Procesando 10 anuncios...
Página 21: Procesando 10 anuncios...
Página 22: Procesando 10 anuncios...
Página 23: Procesando 10 anuncios...
Página 24: Procesando 10 anuncios...
Página 25: Procesando 10 anuncios...
Página 26: Procesando 10 anuncios...
Página 27: Procesando 10 anuncios...
Página 28: